In [5]:
import sys
from pathlib import Path
raiz_proyecto = Path().resolve().parent
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from src.data_repository import DataRepository

# Desactivar advertencias visuales para mantener limpia la consola/salida
warnings.filterwarnings('ignore')

# Definir la semilla aleatoria global para reproducibilidad técnica
SEED = 42
np.random.seed(SEED)

print("=== 1. DESCRIPCIÓN GENERAL DEL DATASET ===")

# 1. Construir las rutas absolutas combinando la raíz con las subcarpetas del repo
ruta_datos_crudos = raiz_proyecto / 'data' / 'raw' / 'incidencia_delictiva.csv'
ruta_datos_limpios = raiz_proyecto / 'data' / 'processed' / 'clean.csv'

# 2. Instanciar tu patrón de diseño Repository con las rutas corregidas
repo = DataRepository.create(
    raw_path=ruta_datos_crudos, 
    processed_path=ruta_datos_limpios,
    encoding='latin-1'
)

# 3. Cargar los datos
df = repo.load_raw()

print(f"Dataset cargado con éxito.")
print(f"Dimensiones físicas: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print("\nEstructura de tipos de datos por variable:")
df.info()

=== 1. DESCRIPCIÓN GENERAL DEL DATASET ===
Dataset cargado con éxito.
Dimensiones físicas: 413,952 filas × 11 columnas

Estructura de tipos de datos por variable:
<class 'pandas.DataFrame'>
RangeIndex: 413952 entries, 0 to 413951
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   anio                    413952 non-null  int64
 1   clave_ent               413952 non-null  int64
 2   entidad                 413952 non-null  str  
 3   bien_juridico_afectado  413952 non-null  str  
 4   tipo_delito             413952 non-null  str  
 5   subtipo_delito          413952 non-null  str  
 6   modalidad               413952 non-null  str  
 7   mes                     413952 non-null  str  
 8   fecha                   413952 non-null  str  
 9   incidencia_delictiva    413952 non-null  int64
 10  entidad_federativa      413952 non-null  str  
dtypes: int64(3), str(8)
memory usage: 34.7 MB


In [6]:
print("=== 2. CALIDAD DE DATOS ===")

# 1. Análisis de valores faltantes (conteo y proporción porcentual)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    "Cantidad_de_valores_faltantes": missing.values,
    "Porcentaje (%)": missing_pct.values
}, index=missing.index).query("Cantidad_de_valores_faltantes > 0").sort_values("Porcentaje (%)", ascending=False)

if not missing_df.empty:
    print("Métricas de Valores Faltantes por Columna:")
    print(missing_df.to_string())
else:
    print("Calidad de datos validada: No se encontraron valores faltantes.")

# 2. Análisis de redundancia por registros duplicados
columna_cotejo = [
    'anio', 'entidad', 'bien_juridico_afectado', 'tipo_delito', 
    'subtipo_delito', 'modalidad', 'mes', 'fecha', 'incidencia_delictiva', 'entidad_federativa'
]
n_duplicados = df.duplicated(subset=columna_cotejo).sum()
print(f"\nCantidad de registros duplicados detectados: {n_duplicados:,}")

=== 2. CALIDAD DE DATOS ===
Calidad de datos validada: No se encontraron valores faltantes.

Cantidad de registros duplicados detectados: 0


In [ ]:
print("=== 3. ESTADÍSTICAS DESCRIPTIVAS ===")

print("\n--- Distribución Cronológica por Año ---")
print(df['anio'].value_counts().sort_index())

print("\n--- Distribución Estacional por Mes ---")
orden_meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
               'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
print(df['mes'].value_counts().reindex(orden_meses))

print("\n--- Análisis de Variables Categóricas (Top 10 y Moda) ---")
cat_cols = ['entidad', 'bien_juridico_afectado', 'tipo_delito', 'subtipo_delito', 'modalidad']

for col in cat_cols:
    print(f"\nVariable: {col} | Moda: {df[col].mode()[0]}")
    freq = df[col].value_counts().head(10)
    freq_rel = df[col].value_counts(normalize=True).mul(100).round(2).head(10)
    
    tabla_desc = pd.DataFrame({'Frecuencia Absoluta': freq, 'Porcentaje (%)': freq_rel})
    print(tabla_desc)

In [ ]:
# Configuración del entorno gráfico institucional
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("=== 4. DETECCIÓN DE VALORES ATÍPICOS ===")

# 1. Distribución de la Incidencia Delictiva por Entidad Federativa
plt.figure(figsize=(14, 8))
sns.boxplot(data=df, x='entidad_federativa', y='incidencia_delictiva')
plt.xticks(rotation=90)
plt.title('Distribución de la Incidencia Delictiva por Entidad Federativa', fontsize=14, fontweight='bold')
plt.xlabel('Entidad Federativa')
plt.ylabel('Incidencia Delictiva')
plt.tight_layout()
plt.show()

# 2. Distribución de Incidencia de los 10 Delitos más Frecuentes
delitos_top = df.groupby('tipo_delito')['incidencia_delictiva'].sum().sort_values(ascending=False).head(10).index
df_top_delitos = df[df['tipo_delito'].isin(delitos_top)]

plt.figure(figsize=(14, 8))
sns.boxplot(data=df_top_delitos, x='tipo_delito', y='incidencia_delictiva')
plt.xticks(rotation=45, ha='right')
plt.title('Distribución de Incidencia de los 10 Delitos más Frecuentes', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de Delito')
plt.ylabel('Incidencia Delictiva')
plt.tight_layout()
plt.show()

# 3. Distribución de Incidencia por Bien Jurídico Afectado
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='bien_juridico_afectado', y='incidencia_delictiva')
plt.xticks(rotation=45, ha='right')
plt.title('Distribución de Incidencia por Bien Jurídico Afectado', fontsize=14, fontweight='bold')
plt.xlabel('Bien Jurídico Afectado')
plt.ylabel('Incidencia Delictiva')
plt.tight_layout()
plt.show()

# 4. Distribución de Incidencia Delictiva por Mes
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='mes', y='incidencia_delictiva', order=orden_meses)
plt.xticks(rotation=45)
plt.title('Distribución de Incidencia Delictiva por Mes', fontsize=14, fontweight='bold')
plt.xlabel('Mes')
plt.ylabel('Incidencia Delictiva')
plt.tight_layout()
plt.show()

# 5. Distribución de Incidencia por Entidad y Tipo de Delito (Top 5 Estados - Robo)
entidades_top = df.groupby('entidad_federativa')['incidencia_delictiva'].sum().sort_values(ascending=False).head(5).index
df_comparacion = df[(df['entidad_federativa'].isin(entidades_top)) & (df['tipo_delito'].isin(['Robo']))]

plt.figure(figsize=(14, 8))
sns.boxplot(data=df_comparacion, x='entidad_federativa', y='incidencia_delictiva', hue='tipo_delito')
plt.xticks(rotation=45)
plt.title('Distribución de Incidencia por Entidad y Tipo de Delito (Robo)', fontsize=14, fontweight='bold')
plt.xlabel('Entidad Federativa')
plt.ylabel('Incidencia Delictiva')
plt.legend(title='Tipo de Delito', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 6. Distribución de Incidencia Delictiva por Año
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='anio', y='incidencia_delictiva')
plt.title('Distribución de Incidencia Delictiva por Año', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Incidencia Delictiva')
plt.tight_layout()
plt.show()

> ### 📝 Interpretación Analítica de los Diagramas de Caja (Outliers)
> * **Qué se observa:** Los diagramas de caja revelan una dispersión asimétrica extrema en la variable `incidencia_delictiva` al segmentarse por bien jurídico. La categoría *El patrimonio* presenta la mediana más elevada y un rango intercuartílico (IQR) notablemente ancho, acompañada de una densa concentración de valores atípicos en la parte superior. Al analizar el comportamiento macro a lo largo de los años (2015-2025), las cajas mantienen una posición y un IQR estables en el tiempo, confirmando que la estructura general de la denuncia no sufre mutaciones drásticas, aunque persisten sistemáticamente los puntos atípicos en todos los periodos.
> * **Implicación para el modelado:** La drástica diferencia en las escalas demuestra que una baja incidencia absoluta no significa la ausencia de patrones delictivos, sino que responde a la naturaleza intrínseca del delito (los robos ocurren con magnitudes numéricas masivas, mientras que los secuestros registran conteos bajos pero críticos). Debido a que estos outliers no corresponden a errores de captura, sino a picos reales de criminalidad concentrados en regiones de alta densidad poblacional, la estrategia de limpieza **no debe eliminar estos registros**. Su conservación es fundamental para que el modelo aprenda a identificar escenarios de riesgo extremo.

In [ ]:
print("=== 6. RELACIONES ENTRE VARIABLES ===")

delitos_interes = [
    'Robo de maquinaria', 'Robo a negocio', 'Robo de vehículo automotor', 'Homicidio culposo',
    'Secuestro', 'Feminicidio', 'Lesiones culposas', 'Violencia familiar',
    'Homicidio doloso', 'Abuso sexual', 'Extorsión', 'Daño a la propiedad'
]

# Filtrar y pivotar para estructurar la matriz estado × delito
df_filtrado = df[df['subtipo_delito'].isin(delitos_interes)].copy()
tabla_incidencia = df_filtrado.groupby(['entidad_federativa', 'tipo_delito'])['incidencia_delictiva'].sum().reset_index()
tabla_pivot = tabla_incidencia.pivot(index='entidad_federativa', columns='tipo_delito', values='incidencia_delictiva').fillna(0)

# Calcular matriz usando coeficientes de Spearman
matriz_correlacion = tabla_pivot.corr(method='spearman')

# Visualización mediante Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz_correlacion, 
    annot=True, 
    cmap='coolwarm', 
    center=0, 
    vmin=-1, vmax=1, 
    square=True, 
    linewidths=0.5, 
    cbar_kws={"shrink": 0.8}
)
plt.title('Matriz de Correlación de Spearman entre Diferentes Tipos de Delito por Estado', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

> ### 📝 Interpretación de la Relación entre Variables (Mapa de Calor)
> * **Qué se observa:** El mapa de calor basado en coeficientes de Spearman muestra correlaciones positivas fuertes (valores mayores a 0.70) entre delitos específicos como *Robo a negocio*, *Robo de vehículo automotor* y *Extorsión*. Por otro lado, delitos de alto impacto violento como *Homicidio doloso* y *Feminicidio* exhiben una covariación estrecha entre sí, pero una correlación moderada con los delitos de carácter puramente patrimonial.
> * **Implicación para el modelado:** La fuerte coexistencia de ciertas tipologías delictivas a nivel geográfico demuestra que las entidades federativas no sufren de criminalidad aislada, sino de perfiles estructurales de inseguridad de carácter regional. Este hallazgo valida por completo la pertinencia de aplicar técnicas de agrupamiento no supervisado (*Clustering*) en la siguiente fase para clasificar los estados en regiones homogéneas de riesgo.

In [ ]:
print("=== 7. ANÁLISIS DE VARIABLES CATEGÓRICAS ===")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Por año
por_anio = df.groupby('anio')['incidencia_delictiva'].sum().reset_index()
sns.barplot(x='anio', y='incidencia_delictiva', data=por_anio, ax=axes[0,0], color="#1a6fd4")
axes[0,0].set_title("Incidencia por Año", fontsize=12, fontweight='bold')
axes[0,0].set_xlabel("Año")
axes[0,0].set_ylabel("Total de casos")
axes[0,0].tick_params(axis='x', rotation=45)

# 2. Por mes
por_mes = df.groupby('mes')['incidencia_delictiva'].sum().reindex(orden_meses).reset_index()
sns.barplot(x='mes', y='incidencia_delictiva', data=por_mes, ax=axes[0,1], color="#c8a951")
axes[0,1].set_title("Incidencia por Mes", fontsize=12, fontweight='bold')
axes[0,1].set_xlabel("Mes")
axes[0,1].set_ylabel("Total de casos")
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Por estado (top 15)
por_estado = df.groupby('entidad_federativa')['incidencia_delictiva'].sum().sort_values(ascending=False).head(15).reset_index()
sns.barplot(x='incidencia_delictiva', y='entidad_federativa', data=por_estado, ax=axes[1,0], color="#2ecc71")
axes[1,0].set_title("Top 15 Estados con Mayor Incidencia", fontsize=12, fontweight='bold')
axes[1,0].set_xlabel("Total de casos")
axes[1,0].set_ylabel("Estado")

# 4. Por tipo de delito (top 15)
por_delito = df.groupby('subtipo_delito')['incidencia_delictiva'].sum().sort_values(ascending=False).head(15).reset_index()
sns.barplot(x='incidencia_delictiva', y='subtipo_delito', data=por_delito, ax=axes[1,1], color="#e74c3c")
axes[1,1].set_title("Top 15 Tipos de Delito", fontsize=12, fontweight='bold')
axes[1,1].set_xlabel("Total de casos")
axes[1,1].set_ylabel("Tipo de delito")

plt.suptitle("Análisis Macroscópico de Incidencia Delictiva", y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> ### 📝 Interpretación del Análisis Volumétrico Categórico
> * **Qué se observa:** El panel de barras distribuidas refleja tres fenómenos claros: una tendencia macroeconómica ligeramente ascendente por año, un comportamiento cíclico estacional por mes que registra caídas drásticas en febrero y repuntes constantes hacia el cierre de año (noviembre/diciembre), y una concentración delictiva masiva donde un grupo reducido de estados absorbe la mayoría de las denuncias del país.
> * **Implicación para el modelado:** La marcada estacionalidad mensual y la evolución anual demuestran que el factor tiempo tiene un peso predictivo real en la fluctuación de la delincuencia. La enorme disparidad entre estados confirma que la geografía es un factor discriminante clave, lo que justifica la inclusión formal de `anio`, `mes_num` y `clave_ent` como variables de entrada (*features*) para el clasificador.

In [ ]:
print("=== 8. ANÁLISIS DE SERIES DE TIEMPO ===")

# Validar y formatear campos de series de tiempo deducidos
if 'temporal_fecha' in df.columns:
    df['temporal_fecha'] = pd.to_datetime(df['temporal_fecha'])
    df_ts = df.groupby('temporal_fecha')['incidencia_delictiva'].sum().reset_index().sort_values('temporal_fecha')

    plt.figure(figsize=(14, 6))
    plt.plot(df_ts['temporal_fecha'], df_ts['incidencia_delictiva'], color='#003f8a', linewidth=2, marker='.', markersize=4)
    plt.title('Evolución Temporal Macroscópica de la Incidencia Delictiva Total (2015-2025)', fontsize=14, fontweight='bold')
    plt.xlabel('Línea de Tiempo Histórica')
    plt.ylabel('Volumen Agregado de Delitos Registrados')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

> ### 📝 Interpretación Dinámica de la Serie de Tiempo Histórica
> * **Qué se observa:** La línea de tiempo continua (2015-2025) mapea la trayectoria delictiva agregada del país. Se aprecian fluctuaciones periódicas regulares y quiebres estructurales específicos en la serie temporal, destacando contracciones atípicas que coinciden históricamente con periodos de reducción de movilidad social o confinamientos a nivel nacional.
> * **Implicación para el modelado:** Este comportamiento dinámico confirma que el fenómeno delictivo no es estacionario y responde a factores externos del entorno. Para que el modelo supervisado pueda capturar de manera correcta las tendencias macro sin sufrir de *data leakage* (fuga de datos), se deben omitir las variables de subcategorías directas (como el tipo o subtipo de delito), y permitir que el algoritmo aprenda exclusivamente de los componentes contextuales de la serie histórica.